Conectado a venv (Python 3.11.9)

In [ ]:
import pandas as pd 
import numpy as np 
import xgboost as xgb  # Importando o XGBoost
from sklearn.pipeline import Pipeline
from modulos.model_pre_processing import importar_dados, preprocessar_dados
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error
from scipy.stats import randint
import joblib

In [ ]:
# Utilizando as funções criadas para importar e pré-processar os dados
X_train, X_val, X_test, y_train, y_val, y_test = importar_dados() 
preprocessor = preprocessar_dados()
X_train.head()

Tamanho do treino: (31848, 5)
Tamanho da validação: (10616, 5)
Tamanho do teste: (10616, 5)


,ANO_VENCIMENTO,MES_VENCIMENTO,TRIMESTRE,VALOR_FATURA_lag1,VALOR_FATURA_lag2
0,2022,10,4,39.22,28.02
1,2023,1,1,156.26,101.46
2,2023,4,2,280.17,123.47
3,2024,12,4,56.22,63.44
4,2023,3,1,45.62,74.75


In [ ]:
# Criando o modelo de regressão XGBoost
# O 'random_state=42' garante que os resultados sejam reprodutíveis.
model = xgb.XGBRegressor(random_state=42)

In [ ]:
# Pipeline
# O pipeline permite integrar o pré-processamento e o modelo em um único fluxo de trabalho.
# Isso facilita a execução do pré-processamento e a aplicação do modelo de forma sequencial e correta.
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # Primeiro, aplica o pré-processamento aos dados.
    ('regressor', model)  # Depois, treina o modelo XGBoost.
])

In [ ]:
param_grid = {
    'regressor__colsample_bytree': [0.3, 0.5, 0.7],
    'regressor__max_depth': [3, 5, 7],
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__n_estimators': [50, 100, 200]
}

# GridSearchCV testa todas as combinações possíveis dos hiperparâmetros definidos acima.
grid_search = GridSearchCV(
    pipeline,  # O pipeline com o pré-processamento e o modelo.
    param_grid,  # O espaço de busca para os hiperparâmetros.
    cv=5,  # Número de divisões (folds) para validação cruzada.
    scoring='neg_mean_absolute_error',  # A métrica a ser otimizada.
    n_jobs=-1  # Utiliza todos os núcleos de processamento disponíveis para acelerar a busca.
)

grid_search.fit(X_train, y_train)  # Ajusta o GridSearchCV aos dados de treino

print("Melhores hiperparâmetros:", grid_search.best_params_)
print("Melhor score:", grid_search.best_score_)

Melhores hiperparâmetros: {'regressor__colsample_bytree': 0.3, 'regressor__learning_rate': 0.2, 'regressor__max_depth': 3, 'regressor__n_estimators': 50}
Melhor score: -266.3429284636577


In [ ]:
# Após a busca de hiperparâmetros, obtemos o modelo final treinado com os melhores parâmetros.
xgboost_model = grid_search.best_estimator_

 ### Avaliando Resultado dos modelos

In [ ]:
# Avalia o modelo com a validação cruzada
validacao_cruzada = KFold(n_splits=10, shuffle=True, random_state=42)
cross_val_scores = cross_val_score(xgboost_model, X_val, y_val, cv=validacao_cruzada)
acuracia_media_xgb = cross_val_scores.mean()
print(cross_val_scores)
print("Acurácia média do XGBoost:", acuracia_media_xgb)

# Faz previsões com o pipeline ajustado
y_pred = xgboost_model.predict(X_val)

[0.74186152 0.72320914 0.69962567 0.70949996 0.69147146 0.68693799
 0.68122667 0.72009981 0.77864635 0.68735701]
Acurácia média do XGBoost: 0.7119935572147369


 1. Mean Absolute Error (MAE)
 O MAE calcula o erro médio absoluto entre as previsões e os valores reais. Ele é útil para entender o erro médio sem considerar a direção (positivo ou negativo).

In [ ]:
from sklearn.metrics import mean_absolute_error

# y_val: valores reais, y_pred: valores previstos
mae = mean_absolute_error(y_val, y_pred)

# Exibindo o valor do MAE
print(f"Mean Absolute Error (MAE): {mae}")

Mean Absolute Error (MAE): 278.70664815694295


 2. Mean Squared Error (MSE)
 O MSE calcula o erro quadrado médio. Ele penaliza mais os erros grandes, pois a diferença entre o valor real e o previsto é elevada ao quadrado

In [ ]:
from sklearn.metrics import mean_squared_error

# y_val: valores reais, y_pred: valores previstos
mse = mean_squared_error(y_val, y_pred)

# Exibindo o valor do MSE
print(f"Mean Squared Error (MSE): {mse}")

Mean Squared Error (MSE): 323331.84996168205


 3. Root Mean Squared Error (RMSE)
 O RMSE é a raiz quadrada do MSE. Ele traz a medida do erro para a mesma escala dos dados originais, o que torna mais fácil de interpretar.

In [ ]:
import numpy as np
# Calculando o RMSE como a raiz quadrada do MSE
rmse = np.sqrt(mse)

# Exibindo o valor do RMSE
print(f"Root Mean Squared Error (RMSE): {rmse}")

Root Mean Squared Error (RMSE): 568.6227659544437


 4. Coefficient of Determination (R²)
 O R² (ou coeficiente de determinação) mede a proporção da variação nos dados que o modelo é capaz de explicar. Um valor de 1 significa explicação perfeita, enquanto um valor de 0 significa que o modelo não explicou nada além da média.

In [ ]:
from sklearn.metrics import r2_score

# y_val: valores reais, y_pred: valores previstos
r2 = r2_score(y_val, y_pred)

# Exibindo o valor do R²
print(f"Coefficient of Determination (R²): {r2}")

Coefficient of Determination (R²): 0.7164933681488037


 5. Explained Variance Score
 A Expained Variance Score calcula a variação explicada pelo modelo. Um valor próximo de 1 significa que o modelo explicou a maior parte da variação dos dados.

In [ ]:
from sklearn.metrics import explained_variance_score

# y_val: valores reais, y_pred: valores previstos
explained_variance = explained_variance_score(y_val, y_pred)

# Exibindo o valor do Explained Variance Score
print(f"Explained Variance Score: {explained_variance}")

Explained Variance Score: 0.7165629218070972


In [ ]:
# Após treinar o modelo e validá-lo, podemos salvar o modelo final em um arquivo para uso posterior.
# O arquivo será salvo com o nome 'xgboost_model.joblib' para fácil carregamento e uso no futuro.
joblib.dump(xgboost_model, 'xgboost_model.joblib')

['xgboost_model.joblib']